# 본 분석 전 기술통계 및 전처리 필요 항목 점검

본 분석에선 실제 전처리를 수행하지 않고,\
**분석에 들어가기 앞서 어떤 전처리가 필요한지 확인하는 진단용 기술통계**입니다.

## 중점 확인 항목
| 기준 | 확인 내용 |
|---|---|
| 결측 | 컬럼별 결측치 수와 비율 |
| 중복 | 완전 중복 행, `appid` 중복 |
| 공백 | 문자열 컬럼의 빈 문자열, 앞뒤 공백, 연속 공백 |
| 타입 | 컬럼별 dtype, 고유값 수, 구조형 문자열 여부 |
| 이상치 | 수치형 컬럼의 IQR 기준 이상치 후보 |
| 조인 가능성 | `appid` 기준 테이블 간 매칭률 |


## 점검 범위

### 메인 기술통계 대상
- `steam_indie_list_202604211615.csv`
- `steam_indie_9692_202604281029.csv`
- `steam_app_details_202604281555.csv`

### 보조 확인 대상
- `steam_indie_reviews_202604230927.csv`
- `steam_indie_tags_202604281545.csv`


> 주의: 여기서 이상치는 삭제 대상이 아니라 **확인 후보**입니다. \Steam 데이터는 리뷰 수, 동시접속자 수, 소유자 수가 자연스럽게 긴 꼬리 분포를 가질 수 있습니다.

# 1. 라이브러리 불러오기

In [84]:
from pathlib import Path
import ast
import json
import re
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 보기 옵션
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# 2. 파일 경로 설정 및 데이터 읽기


In [85]:
# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정
# db에서 바로 불러오는법을 모름
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# 원천/소스 파일이 들어있는 폴더
DATA_DIR = ROOT / "data" / "processed"

# 파일 경로
## 메인 기술통계 대상
INDIE_LIST_PATH = DATA_DIR / "steam_indie_list_202604211615.csv"
INDIE_9692_PATH = DATA_DIR / "steam_indie_9692_202604281029.csv"
APP_DETAILS_PATH = DATA_DIR / "steam_app_details_202604281555.csv"

## 보조 확인 대상
REVIEWS_PATH = DATA_DIR / "steam_indie_reviews_202604230927.csv"
TAGS_PATH = DATA_DIR / "steam_indie_tags_202604281545.csv"


# 경로 확인
# 파일이 없으면 이후 read_csv 단계에서 에러가 나므로, 먼저 exists() 결과를 확인한다.
print("ROOT             =", ROOT)
print("DATA_DIR         =", DATA_DIR)
print("INDIE_LIST_PATH  =", INDIE_LIST_PATH)
print("INDIE_9692_PATH  =", INDIE_9692_PATH)
print("APP_DETAILS_PATH =", APP_DETAILS_PATH)
print("REVIEWS_PATH     =", REVIEWS_PATH)
print("TAGS_PATH        =", TAGS_PATH)

print()
print("indie_list exists  :", INDIE_LIST_PATH.exists())
print("indie_9692 exists  :", INDIE_9692_PATH.exists())
print("app_details exists :", APP_DETAILS_PATH.exists())
print("reviews exists     :", REVIEWS_PATH.exists())
print("tags exists        :", TAGS_PATH.exists())

# 데이터 읽기
# 여기서는 원본을 바로 전처리하지 않고, 파일을 읽어온 뒤 별도 확인용 복사본을 만든다.
indie_list_df = pd.read_csv(INDIE_LIST_PATH)
indie_9692_df = pd.read_csv(INDIE_9692_PATH)
app_details_df = pd.read_csv(APP_DETAILS_PATH)

reviews_df = pd.read_csv(REVIEWS_PATH)
tags_df = pd.read_csv(TAGS_PATH)

# 크기 확인
print()
print("indie_list_df shape  :", indie_list_df.shape)
print("indie_9692_df shape  :", indie_9692_df.shape)
print("app_details_df shape :", app_details_df.shape)
print("reviews_df shape     :", reviews_df.shape)
print("tags_df shape        :", tags_df.shape)

ROOT             = C:\Users\joon5\Documents\github\steam-indie-game-analysis
DATA_DIR         = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed
INDIE_LIST_PATH  = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_list_202604211615.csv
INDIE_9692_PATH  = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_9692_202604281029.csv
APP_DETAILS_PATH = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_app_details_202604281555.csv
REVIEWS_PATH     = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_reviews_202604230927.csv
TAGS_PATH        = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_tags_202604281545.csv

indie_list exists  : True
indie_9692 exists  : True
app_details exists : True
reviews exists     : True
tags exists        : True

indie_list_df shape  : (61266, 12)
indie_9692_df shape  : (9692, 

In [86]:
# 컬럼 목록 확인
print('[indie_list_df columns]')
print(indie_list_df.columns.tolist())
print()

print('[indie_9692_df columns]')
print(indie_9692_df.columns.tolist())
print()

print('[app_details_df columns]')
print(app_details_df.columns.tolist())
print()

print('[reviews_df columns]')
print(reviews_df.columns.tolist())
print()

print('[tags_df columns]')
print(tags_df.columns.tolist())

[indie_list_df columns]
['appid', 'spy_name', 'owners', 'positive', 'negative', 'price_spy', 'ccu', 'name_store', 'type', 'genres', 'release_date', 'developers']

[indie_9692_df columns]
['appid', 'name', 'owners', 'positive', 'negative', 'price', 'ccu', 'genres', 'release_date', 'developers', 'total_reviews', 'owners_lower', 'is_f2p', 'is_early_access']

[app_details_df columns]
['appid', 'name', 'type', 'is_free', 'controller_support', 'short_description', 'supported_languages', 'developers', 'publishers', 'genres', 'categories', 'coming_soon', 'release_date', 'currency', 'initial', 'final', 'discount_percent', 'initial_formatted', 'final_formatted', 'windows', 'mac', 'linux', 'recommendations_total', 'metacritic_score', 'metacritic_url', 'achievements_total', 'header_image', 'website', 'collected_at']

[reviews_df columns]
['recommendationid', 'appid', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comme

# 3. 점검 대상 데이터

| 구분 | 파일명 | 역할 | 점검 수준 |
|---|---|---|---|
| 메인 | `steam_indie_list_202604211615.csv` | 전체 인디게임 모집단 | 상세 점검 |
| 메인 | `steam_indie_9692_202604281029.csv` | 본 분석 후보군 | 상세 점검 |
| 메인 | `steam_app_details_202604281555.csv` | 게임 상세 메타데이터 | 상세 점검 |
| 보조 | `steam_indie_reviews_202604230927.csv` | 미니 리뷰 분석용 데이터 | 타입/조인만 확인 |
| 보조 | `steam_indie_tags_202604281545.csv` | 미니 태그 분석용 데이터 | 타입/조인만 확인 |

# 4. 기술통계 확인을 위한 최소 보조 함수
제 전처리를 완료하기 위한 함수가 아닌,\  
전처리 필요 여부를 확인하기 위한 임시 점검 함수

In [87]:
def parse_list_like(value):
    """리스트처럼 저장된 문자열 또는 쉼표 구분 문자열을 기술통계 확인용 리스트로 변환"""
    if pd.isna(value):
        return []
    
    text = str(value).strip()

    if text == "" or text.lower() in ["nan", "none"]:
        return []
    
    # ['Action', 'Indie'] 같은 형태일 때만 literal_eval 시도
    if text[0] in "[({":
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return parsed
            if isinstance(parsed, tuple):
                return list(parsed)
            if isinstance(parsed, dict):
                return list(parsed.keys())
        except (ValueError, SyntaxError):
            pass
    # 예: "Action, Indie"처럼 쉼표 구분 문자열로 저장된 경우
    if "," in text:
        return [x.strip() for x in text.split(",") if x.strip() != ""]
    # 단일 값이면 리스트 하나로 감쌈
    return [text]

def parse_dict_like(value):
    """딕셔너리처럼 저장된 문자열을 기술통계 확인용 dict로 변환"""
    if pd.isna(value):
        return {}
    text = str(value).strip()
    
    if text == "" or text.lower() in ["nan", "none"]:
        return {}
    
    if text[0] == "{":
        # JSON 형태 우선 시도
        try:
            parsed = json.loads(text)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass
        
        # Python dict 문자열 형태도 보조적으로 시도
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, dict):
                return parsed
        except (ValueError, SyntaxError):
            pass
            
    return {}

def extract_owner_bounds(value):
    """owners 범위 문자열에서 하한/상한/중간값을 기술통계 확인용으로 추출"""
    if pd.isna(value):
        return np.nan, np.nan, np.nan
    
    nums = re.findall(r"[\d,]+", str(value))
    nums = [int(x.replace(",", "")) for x in nums]
    
    if len(nums) >= 2:
        lower = nums[0]
        upper = nums[1]
        mid = (lower + upper) / 2
        return lower, upper, mid
    
    return np.nan, np.nan, np.nan


def parse_dates_for_check(series):
    """여러 날짜 문자열 형식을 최대한 안전하게 확인용 datetime으로 변환한다."""
    try:
        # pandas 2.x에서는 혼합 날짜 형식에 format='mixed'가 더 안정적
        return pd.to_datetime(series, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        # pandas 버전에 따라 format='mixed'가 지원되지 않을 수 있으므로 기본 방식으로 재시도
        return pd.to_datetime(series, errors="coerce")


In [88]:
def make_game_stat_view(df, price_col=None):
    """원본을 바꾸지 않고 기술통계 확인용 임시 파생 컬럼만 추가한 복사본 생성"""
    stat_df = df.copy()
    
    # 긍정/부정 리뷰 수가 있으면 전체 리뷰 수와 긍정률을 확인용으로 계산
    if {"positive", "negative"}.issubset(stat_df.columns):
        stat_df["total_reviews_calc"] = stat_df["positive"] + stat_df["negative"]
        stat_df["positive_ratio_pct_calc"] = np.where(
            stat_df["total_reviews_calc"] > 0,
            stat_df["positive"] / stat_df["total_reviews_calc"] * 100,
            np.nan
        )
    # owners는 범위 문자열이므로 숫자형 확인을 위해 하한/상한/중간값을 임시 추출
    if "owners" in stat_df.columns:
        owner_values = stat_df["owners"].apply(extract_owner_bounds)
        stat_df["owners_lower_calc"] = [x[0] for x in owner_values]
        stat_df["owners_upper_calc"] = [x[1] for x in owner_values]
        stat_df["owners_mid_calc"] = [x[2] for x in owner_values]
    # 장르와 카테고리는 항목 개수 확인용으로만 파싱
    if "genres" in stat_df.columns:
        stat_df["genre_count_calc"] = stat_df["genres"].apply(lambda x: len(parse_list_like(x)))
    if "categories" in stat_df.columns:
        stat_df["category_count_calc"] = stat_df["categories"].apply(lambda x: len(parse_list_like(x)))
    # 가격 컬럼이 있으면 0원/무료 여부를 확인용으로 계산
    if price_col is not None and price_col in stat_df.columns:
        stat_df["is_free_by_price_calc"] = stat_df[price_col].eq(0)

    return stat_df

In [89]:
# 기술통계 확인용 복사본 생성
# 원본 DataFrame은 그대로 두고, 확인에 필요한 임시 파생 컬럼만 추가한다.
indie_list_stat_df = make_game_stat_view(indie_list_df, price_col="price_spy")
indie_9692_stat_df = make_game_stat_view(indie_9692_df, price_col="price")
app_details_stat_df = make_game_stat_view(app_details_df, price_col="final")

print("기술통계 확인용 복사본 생성 완료")
print("indie_list_stat_df shape  :", indie_list_stat_df.shape)
print("indie_9692_stat_df shape  :", indie_9692_stat_df.shape)
print("app_details_stat_df shape :", app_details_stat_df.shape)

기술통계 확인용 복사본 생성 완료
indie_list_stat_df shape  : (61266, 19)
indie_9692_stat_df shape  : (9692, 21)
app_details_stat_df shape : (161860, 32)


# 5. 기술통계 함수 정의
- 기본 정보/타입/결측 확인
- ID 중복 확인
- 범주형 분포 확인
- 문자열 공백 확인
- 수치형 기술통계 확인
- IQR 기준 이상치 후보 확인
- 날짜 파싱 가능성 확인
- 구조형 문자열 확인
- `appid` 기준 조인 가능성 확인

In [90]:
def check_basic_info(df, df_name, exclude_cols=None):
    """행/열 수, 완전 중복 행, 컬럼별 타입/결측/고유값을 한 번에 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 기본 정보 / 타입 / 결측치 확인")
    print(f"{'='*80}\n")

    df_copied = df.copy()

    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # 리스트나 dict가 들어간 컬럼은 nunique 계산에서 에러가 날 수 있어 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)

    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })

    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df_copied) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df_copied) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    }).sort_values(by=['결측치 개수', '고유값 개수'], ascending=[False, False])

    print("[전체 요약]")
    display(overview_df)

    print("[컬럼별 요약]")
    display(summary_df)

    print("[상위 5행]")
    display(df_copied.head())

In [91]:
def check_id_duplicates(df, col_name, df_name, top_n=10):
    """appid처럼 기준 키로 쓸 컬럼의 중복 여부를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 중복 확인")
    print(f"{'='*80}")

    df_copied = df.copy()

    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    duplicate_count = df_copied[col_name].duplicated().sum()

    print('전체 행 수:', len(df_copied))
    print(f'{col_name} 고유 개수:', df_copied[col_name].nunique(dropna=True))
    print(f'중복 {col_name} 개수:', duplicate_count)

    if duplicate_count > 0:
        print()
        print('[중복 상위 값]')
        dup_summary = df_copied[col_name].value_counts(dropna=False).reset_index()
        dup_summary.columns = [col_name, '등장 횟수']
        display(dup_summary[dup_summary['등장 횟수'] > 1].head(top_n))
    else:
        print('중복 값이 없습니다.')

In [92]:
def check_category_summary(df, df_name, col_name, top_n=10):
    """범주형 컬럼의 값 분포를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    summary_df = df[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, "개수"]
    summary_df["비율(%)"] = (summary_df["개수"] / len(df) * 100).round(2)

    print("전체 행 수:", len(df))
    print(f"{col_name} 고유값 개수(결측 포함):", df[col_name].nunique(dropna=False))
    print()

    display(summary_df.head(top_n))

In [93]:
def check_many_categories(df, df_name, cols, top_n=10):
    """여러 범주형 컬럼을 같은 형식으로 반복 확인"""
    for col in cols:
        if col in df.columns:
            check_category_summary(df, df_name, col, top_n=top_n)
        else:
            print(f"{df_name}에 '{col}' 컬럼이 없습니다.")

In [94]:
def check_string_space_summary(df, df_name, top_n=30):
    """문자열 컬럼의 빈 문자열, 앞뒤 공백, 연속 공백을 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 문자열 공백/빈값 확인")
    print(f"{'='*80}")

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()

    if len(object_cols) == 0:
        print("문자열 컬럼이 없습니다.")
        return

    rows = []

    for col in object_cols:
        temp = df[col]
        temp_str = temp.dropna().astype(str)

        empty_count = temp_str.str.strip().eq("").sum()
        leading_trailing_count = temp_str.ne(temp_str.str.strip()).sum()
        multi_space_count = temp_str.str.contains(r"\s{2,}", regex=True).sum()

        rows.append({
            "컬럼명": col,
            "문자열 행 수": len(temp_str),
            "빈 문자열/공백값 개수": empty_count,
            "앞뒤 공백 개수": leading_trailing_count,
            "연속 공백 포함 개수": multi_space_count
        })

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values(
        by=["빈 문자열/공백값 개수", "앞뒤 공백 개수", "연속 공백 포함 개수"],
        ascending=False
    )

    display(summary_df.head(top_n))

In [95]:
def check_numeric_summary(df, df_name, cols=None):
    """수치형 컬럼의 기본 기술통계, 왜도, 첨도를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 수치형 기술통계")
    print(f"{'='*80}")

    if cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        numeric_cols = [col for col in cols if col in df.columns]

    if len(numeric_cols) == 0:
        print("수치형 컬럼이 없습니다.")
        return

    summary_df = df[numeric_cols].describe().T
    summary_df["결측치 개수"] = df[numeric_cols].isnull().sum()
    summary_df["왜도"] = df[numeric_cols].skew(numeric_only=True)
    summary_df["첨도"] = df[numeric_cols].kurt(numeric_only=True)

    display(summary_df)

In [96]:
def check_date_parse_summary(df, df_name, col_name):
    """문자열 날짜 컬럼이 datetime으로 얼마나 변환 가능한지 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 날짜 파싱 가능성 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    temp = df[col_name].dropna()
    parsed = parse_dates_for_check(temp)

    success_count = parsed.notna().sum()
    fail_count = parsed.isna().sum()

    summary_df = pd.DataFrame({
        "항목": [
            "전체 행 수",
            "결측 제외 행 수",
            "날짜 변환 성공 수",
            "날짜 변환 실패 수",
            "날짜 변환 성공률(%)",
            "변환 가능 최소 날짜",
            "변환 가능 최대 날짜"
        ],
        "값": [
            len(df),
            len(temp),
            success_count,
            fail_count,
            round(success_count / len(temp) * 100, 2) if len(temp) > 0 else np.nan,
            parsed.min(),
            parsed.max()
        ]
    })

    display(summary_df)

    if fail_count > 0:
        print("[날짜 변환 실패 예시]")
        fail_examples = temp[parsed.isna()].drop_duplicates().head(10)
        display(fail_examples)


In [97]:
def check_structure_parse_summary(df, df_name, col_name, parse_type="list"):
    """genres, categories, tags처럼 구조형 문자열로 보이는 컬럼의 파싱 가능성을 확인한다."""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 구조형 문자열 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    temp = df[col_name].dropna()

    lengths = []
    success_count = 0

    for value in temp:
        if parse_type == "dict":
            parsed = parse_dict_like(value)
        else:
            parsed = parse_list_like(value)

        parsed_len = len(parsed)
        lengths.append(parsed_len)

        if parsed_len > 0:
            success_count += 1

    fail_or_empty_count = len(temp) - success_count

    summary_df = pd.DataFrame({
        "항목": [
            "결측 제외 행 수",
            "파싱 가능 행 수",
            "빈값/파싱 실패 행 수",
            "파싱 가능 비율(%)",
            "평균 항목 수",
            "중앙값 항목 수",
            "최대 항목 수"
        ],
        "값": [
            len(temp),
            success_count,
            fail_or_empty_count,
            round(success_count / len(temp) * 100, 2) if len(temp) > 0 else np.nan,
            round(np.mean(lengths), 2) if lengths else np.nan,
            np.median(lengths) if lengths else np.nan,
            max(lengths) if lengths else np.nan
        ]
    })

    display(summary_df)

    print("[원본 값 예시]")
    display(temp.head(10))

In [98]:
def check_iqr_outlier_summary(df, df_name, cols=None, top_n=30):
    """IQR 기준으로 이상치 후보가 많은 수치형 컬럼을 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 IQR 기준 이상치 후보 확인")
    print(f"{'='*80}")

    if cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        numeric_cols = [col for col in cols if col in df.columns]

    if len(numeric_cols) == 0:
        print("수치형 컬럼이 없습니다.")
        return

    rows = []

    for col in numeric_cols:
        temp = df[col].dropna()

        if len(temp) == 0:
            continue

        q1 = temp.quantile(0.25)
        q3 = temp.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_count = ((temp < lower) | (temp > upper)).sum()

        rows.append({
            "컬럼명": col,
            "확인 행 수": len(temp),
            "결측치 개수": df[col].isnull().sum(),
            "최솟값": temp.min(),
            "Q1": q1,
            "중앙값": temp.median(),
            "Q3": q3,
            "최댓값": temp.max(),
            "IQR": iqr,
            "하한 기준": lower,
            "상한 기준": upper,
            "이상치 후보 개수": outlier_count,
            "이상치 후보 비율(%)": round(outlier_count / len(temp) * 100, 2)
        })

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values("이상치 후보 개수", ascending=False)

    display(summary_df.head(top_n))

In [99]:
def check_join_summary(left_df, right_df, left_name, right_name, key="appid", top_n=10):
    """기준 키를 사용해 두 테이블이 얼마나 조인 가능한지 확인"""
    print(f"\n{'='*80}")
    print(f"{left_name} → {right_name} 조인 가능성 확인")
    print(f"{'='*80}")

    if key not in left_df.columns:
        print(f"{left_name}에 '{key}' 컬럼이 없습니다.")
        return

    if key not in right_df.columns:
        print(f"{right_name}에 '{key}' 컬럼이 없습니다.")
        return

    left_keys = set(left_df[key].dropna().astype(str))
    right_keys = set(right_df[key].dropna().astype(str))

    matched_keys = left_keys & right_keys
    unmatched_left_keys = left_keys - right_keys

    result_df = pd.DataFrame({
        "항목": [
            f"{left_name} 고유 {key} 수",
            f"{right_name} 고유 {key} 수",
            "매칭되는 key 수",
            f"{left_name} 기준 미매칭 key 수",
            f"{left_name} 기준 매칭률(%)"
        ],
        "값": [
            len(left_keys),
            len(right_keys),
            len(matched_keys),
            len(unmatched_left_keys),
            round(len(matched_keys) / len(left_keys) * 100, 2) if len(left_keys) > 0 else np.nan
        ]
    })

    display(result_df)

    if len(unmatched_left_keys) > 0:
        print(f"[{left_name}에는 있지만 {right_name}에는 없는 {key} 예시]")
        display(pd.DataFrame({key: list(unmatched_left_keys)[:top_n]}))

# 6. 메인 데이터 기술통계 실행
결측/중복/공백/타입/이상치 후보를 모두 확인

## 메인 기술통계 1: `steam_indie_list`
전체 인디게임 모집단 역할을 하는 데이터

In [100]:
check_basic_info(indie_list_df, "steam_indie_list")
check_id_duplicates(indie_list_df, "appid", "steam_indie_list")
check_string_space_summary(indie_list_df, "steam_indie_list")


steam_indie_list의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,61266
1,열 개수,12
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
developers,str,61170,99.84,96,0.16,41840
release_date,str,61233,99.95,33,0.05,4655
spy_name,str,61258,99.99,8,0.01,60883
name_store,str,61263,100.00,3,0.00,60893
appid,int64,61266,100.00,0,0.00,61266
positive,int64,61266,100.00,0,0.00,3840
negative,int64,61266,100.00,0,0.00,1661
genres,str,61266,100.00,0,0.00,1381
ccu,int64,61266,100.00,0,0.00,612
price_spy,int64,61266,100.00,0,0.00,506


[상위 5행]


,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,release_date,developers
0,1623730,Palworld,"50,000,000 .. 100,000,000",358266,22443,2999,18028,Palworld,game,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...","18 Jan, 2024",Pocketpair
1,304930,Unturned,"50,000,000 .. 100,000,000",506516,48852,0,10408,Unturned,game,"['Action', 'Adventure', 'Casual', 'Indie', 'Fr...","7 Jul, 2017",Smartly Dressed Games
2,105600,Terraria,"20,000,000 .. 50,000,000",1373979,35494,999,24580,Terraria,game,"['Action', 'Adventure', 'Indie', 'RPG']","16 May, 2011",Re-Logic
3,431960,Wallpaper Engine,"20,000,000 .. 50,000,000",876898,17560,499,91184,Wallpaper Engine,game,"['Casual', 'Indie', 'Animation & Modeling', 'D...","16 Nov, 2018",Wallpaper Engine Team
4,291550,Brawlhalla,"20,000,000 .. 50,000,000",314809,71647,0,14169,Brawlhalla,game,"['Action', 'Indie', 'Free To Play']","17 Oct, 2017",Blue Mammoth Games



steam_indie_list의 appid 값 중복 확인
전체 행 수: 61266
appid 고유 개수: 61266
중복 appid 개수: 0
중복 값이 없습니다.

steam_indie_list의 문자열 공백/빈값 확인


C:\Users\joon5\AppData\Local\Temp\ipykernel_40888\2408928782.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object"]).columns.tolist()


,컬럼명,문자열 행 수,빈 문자열/공백값 개수,앞뒤 공백 개수,연속 공백 포함 개수
2,name_store,61263,1,29,97
6,developers,61170,1,24,30
0,spy_name,61258,0,37,99
1,owners,61266,0,0,0
3,type,61266,0,0,0
4,genres,61266,0,0,0
5,release_date,61233,0,0,0


In [101]:
check_many_categories(
    indie_list_stat_df,
    "steam_indie_list",
    cols=["type", "is_free_by_price_calc"],
    top_n=20
)

check_structure_parse_summary(indie_list_df, "steam_indie_list", "genres", parse_type="list")
check_date_parse_summary(indie_list_df, "steam_indie_list", "release_date")


steam_indie_list의 type 범주 확인
전체 행 수: 61266
type 고유값 개수(결측 포함): 1



,type,개수,비율(%)
0,game,61266,100.0



steam_indie_list의 is_free_by_price_calc 범주 확인
전체 행 수: 61266
is_free_by_price_calc 고유값 개수(결측 포함): 2



,is_free_by_price_calc,개수,비율(%)
0,False,53986,88.12
1,True,7280,11.88



steam_indie_list의 genres 구조형 문자열 확인


...

0    ['Action', 'Adventure', 'Indie', 'RPG', 'Early...
1    ['Action', 'Adventure', 'Casual', 'Indie', 'Fr...
2              ['Action', 'Adventure', 'Indie', 'RPG']
3    ['Casual', 'Indie', 'Animation & Modeling', 'D...
4                  ['Action', 'Indie', 'Free To Play']
5                    ['Casual', 'Indie', 'Simulation']
6    ['Action', 'Adventure', 'Indie', 'Massively Mu...
7    ['Action', 'Adventure', 'Indie', 'Massively Mu...
8                       ['Indie', 'RPG', 'Simulation']
9       ['Action', 'Adventure', 'Indie', 'Simulation']
Name: genres, dtype: str


steam_indie_list의 release_date 날짜 파싱 가능성 확인


,항목,값
0,전체 행 수,61266
1,결측 제외 행 수,61233
2,날짜 변환 성공 수,61166
3,날짜 변환 실패 수,67
4,날짜 변환 성공률(%),99.89
5,변환 가능 최소 날짜,1997-06-30 00:00:00
6,변환 가능 최대 날짜,2029-11-07 00:00:00


[날짜 변환 실패 예시]


35948        Coming soon
39399    To be announced
Name: release_date, dtype: str

In [102]:
check_numeric_summary(
    indie_list_stat_df,
    "steam_indie_list",
    cols=[
        "positive", "negative", "total_reviews_calc",
        "positive_ratio_pct_calc", "price_spy", "ccu",
        "genre_count_calc",
        "owners_lower_calc", "owners_upper_calc", "owners_mid_calc"
    ]
)

check_iqr_outlier_summary(
    indie_list_stat_df,
    "steam_indie_list",
    cols=[
        "positive", "negative", "total_reviews_calc",
        "positive_ratio_pct_calc", "price_spy", "ccu",
        "genre_count_calc",
        "owners_lower_calc", "owners_upper_calc", "owners_mid_calc"
    ]
)



steam_indie_list의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
positive,61266.0,838.673897,1.414126e+04,0.0,5.000000,17.000000,79.00000,1373979.0,0,53.674034,3741.857666
negative,61266.0,103.772908,1.382147e+03,0.0,1.000000,5.000000,22.00000,156649.0,0,60.598773,5045.758127
total_reviews_calc,61266.0,942.446806,1.509816e+04,0.0,6.000000,23.000000,104.00000,1409473.0,0,52.197823,3544.100505
positive_ratio_pct_calc,60897.0,76.099078,2.357968e+01,0.0,65.260546,82.051282,94.61496,100.0,369,-1.306691,1.523712
price_spy,61266.0,656.000963,1.114242e+03,0.0,109.000000,499.000000,999.00000,99998.0,0,22.403837,1279.497371
ccu,61266.0,25.027079,9.074564e+02,0.0,0.000000,0.000000,0.00000,143870.0,0,102.798253,13481.229886
genre_count_calc,61266.0,3.210508,1.248337e+00,1.0,2.000000,3.000000,4.00000,16.0,0,0.945099,1.862996
owners_lower_calc,61266.0,42538.928606,4.589211e+05,0.0,0.000000,0.000000,20000.00000,50000000.0,0,59.629858,5240.682576
owners_upper_calc,61266.0,107766.624229,1.004997e+06,20000.0,20000.000000,20000.000000,50000.00000,100000000.0,0,55.269531,4227.942318
owners_mid_calc,61266.0,75152.776418,7.309981e+05,10000.0,10000.000000,10000.000000,35000.00000,75000000.0,0,56.454304,4510.278090



steam_indie_list의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
5,ccu,61266,0,0.0,0.000000,0.000000,0.00000,143870.0,0.000000,0.000000,0.000000,11203,18.29
0,positive,61266,0,0.0,5.000000,17.000000,79.00000,1373979.0,74.000000,-106.000000,190.000000,9925,16.20
2,total_reviews_calc,61266,0,0.0,6.000000,23.000000,104.00000,1409473.0,98.000000,-141.000000,251.000000,9808,16.01
1,negative,61266,0,0.0,1.000000,5.000000,22.00000,156649.0,21.000000,-30.500000,53.500000,9331,15.23
8,owners_upper_calc,61266,0,20000.0,20000.000000,20000.000000,50000.00000,100000000.0,30000.000000,-25000.000000,95000.000000,9330,15.23
9,owners_mid_calc,61266,0,10000.0,10000.000000,10000.000000,35000.00000,75000000.0,25000.000000,-27500.000000,72500.000000,9330,15.23
7,owners_lower_calc,61266,0,0.0,0.000000,0.000000,20000.00000,50000000.0,20000.000000,-30000.000000,50000.000000,5447,8.89
3,positive_ratio_pct_calc,60897,369,0.0,65.260546,82.051282,94.61496,100.0,29.354415,21.228924,138.646582,2323,3.81
4,price_spy,61266,0,0.0,109.000000,499.000000,999.00000,99998.0,890.000000,-1226.000000,2334.000000,1621,2.65
6,genre_count_calc,61266,0,1.0,2.000000,3.000000,4.00000,16.0,2.000000,-1.000000,7.000000,319,0.52


## 메인 기술통계 2: `steam_indie_9692`

본 분석 후보군으로 사용하는 데이터

In [103]:
check_basic_info(indie_9692_df, "steam_indie_9692")
check_id_duplicates(indie_9692_df, "appid", "steam_indie_9692")
check_string_space_summary(indie_9692_df, "steam_indie_9692")


steam_indie_9692의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,9692
1,열 개수,14
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
developers,str,9680,99.88,12,0.12,8139
appid,int64,9692,100.00,0,0.00,9692
name,str,9692,100.00,0,0.00,9688
total_reviews,int64,9692,100.00,0,0.00,1427
positive,int64,9692,100.00,0,0.00,1346
release_date,str,9692,100.00,0,0.00,1008
negative,int64,9692,100.00,0,0.00,601
ccu,int64,9692,100.00,0,0.00,294
price,int64,9692,100.00,0,0.00,282
genres,str,9692,100.00,0,0.00,250


[상위 5행]


,appid,name,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access
0,899770,Last Epoch,"20,000,000 .. 50,000,000",88027,22596,3499,5831,"['Action', 'Adventure', 'Indie', 'RPG']",2024-02-21,Eleventh Hour Games,110623,20000000,False,False
1,251570,7 Days to Die,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False
2,1116170,CyberCorp,"10,000,000 .. 20,000,000",266,56,1499,3,"['Action', 'Adventure', 'Indie', 'RPG']",2025-04-22,Megame LLC,322,10000000,False,False
3,1326470,Sons Of The Forest,"10,000,000 .. 20,000,000",222495,31051,2999,4450,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,Endnight Games Ltd,253546,10000000,False,False
4,2186680,"Warhammer 40,000: Rogue Trader","10,000,000 .. 20,000,000",26360,4445,4999,3582,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",2023-12-07,Owlcat Games,30805,10000000,False,False



steam_indie_9692의 appid 값 중복 확인
전체 행 수: 9692
appid 고유 개수: 9692
중복 appid 개수: 0
중복 값이 없습니다.

steam_indie_9692의 문자열 공백/빈값 확인


C:\Users\joon5\AppData\Local\Temp\ipykernel_40888\2408928782.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object"]).columns.tolist()


,컬럼명,문자열 행 수,빈 문자열/공백값 개수,앞뒤 공백 개수,연속 공백 포함 개수
4,developers,9680,0,11,5
0,name,9692,0,7,17
1,owners,9692,0,0,0
2,genres,9692,0,0,0
3,release_date,9692,0,0,0


In [104]:
check_many_categories(
    indie_9692_stat_df,
    "steam_indie_9692",
    cols=["is_f2p", "is_early_access", "is_free_by_price_calc"],
    top_n=20
)

check_structure_parse_summary(indie_9692_df, "steam_indie_9692", "genres", parse_type="list")
check_date_parse_summary(indie_9692_df, "steam_indie_9692", "release_date")


steam_indie_9692의 is_f2p 범주 확인
전체 행 수: 9692
is_f2p 고유값 개수(결측 포함): 1



,is_f2p,개수,비율(%)
0,False,9692,100.0



steam_indie_9692의 is_early_access 범주 확인
전체 행 수: 9692
is_early_access 고유값 개수(결측 포함): 1



,is_early_access,개수,비율(%)
0,False,9692,100.0



steam_indie_9692의 is_free_by_price_calc 범주 확인
전체 행 수: 9692
is_free_by_price_calc 고유값 개수(결측 포함): 2



...

,항목,값
0,결측 제외 행 수,9692.00
1,파싱 가능 행 수,9692.00
2,빈값/파싱 실패 행 수,0.00
3,파싱 가능 비율(%),100.00
4,평균 항목 수,3.12
5,중앙값 항목 수,3.00
6,최대 항목 수,10.00


[원본 값 예시]


0              ['Action', 'Adventure', 'Indie', 'RPG']
1    ['Action', 'Adventure', 'Indie', 'RPG', 'Simul...
2              ['Action', 'Adventure', 'Indie', 'RPG']
3       ['Action', 'Adventure', 'Indie', 'Simulation']
4    ['Action', 'Adventure', 'Indie', 'RPG', 'Strat...
5     ['Adventure', 'Indie', 'Simulation', 'Strategy']
6                     ['Action', 'Adventure', 'Indie']
7    ['Action', 'Adventure', 'Indie', 'Massively Mu...
8                     ['Action', 'Adventure', 'Indie']
9                           ['Action', 'Indie', 'RPG']
Name: genres, dtype: str


steam_indie_9692의 release_date 날짜 파싱 가능성 확인


,항목,값
0,전체 행 수,9692
1,결측 제외 행 수,9692
2,날짜 변환 성공 수,9692
3,날짜 변환 실패 수,0
4,날짜 변환 성공률(%),100.0
5,변환 가능 최소 날짜,2023-01-01 00:00:00
6,변환 가능 최대 날짜,2025-12-27 00:00:00


In [105]:
check_numeric_summary(
    indie_9692_stat_df,
    "steam_indie_9692",
    cols=[
        "positive", "negative", "total_reviews", "total_reviews_calc",
        "positive_ratio_pct_calc", "price", "ccu",
        "genre_count_calc",
        "owners_lower", "owners_lower_calc",
        "owners_upper_calc", "owners_mid_calc"
    ]
)

check_iqr_outlier_summary(
    indie_9692_stat_df,
    "steam_indie_9692",
    cols=[
        "positive", "negative", "total_reviews", "total_reviews_calc",
        "positive_ratio_pct_calc", "price", "ccu",
        "genre_count_calc",
        "owners_lower", "owners_lower_calc",
        "owners_upper_calc", "owners_mid_calc"
    ]
)


steam_indie_9692의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
positive,9692.0,696.681903,6750.596063,0.0,15.000000,34.000000,127.00000,327889.0,0,27.696358,996.277080
negative,9692.0,97.027652,1343.304549,0.0,2.000000,6.000000,22.00000,106084.0,0,57.683397,4171.168253
total_reviews,9692.0,793.709554,7662.793747,10.0,18.000000,41.000000,152.00000,370046.0,0,27.808549,990.715919
total_reviews_calc,9692.0,793.709554,7662.793747,10.0,18.000000,41.000000,152.00000,370046.0,0,27.808549,990.715919
positive_ratio_pct_calc,9692.0,84.181134,15.201690,0.0,77.065297,88.372093,95.34159,100.0,0,-1.473726,2.565816
price,9692.0,883.707078,1058.550707,0.0,299.000000,599.000000,1199.00000,19999.0,0,9.899780,165.728969
ccu,9692.0,35.946038,928.321453,0.0,0.000000,0.000000,1.00000,83936.0,0,78.121949,6915.752078
genre_count_calc,9692.0,3.119480,1.142045,1.0,2.000000,3.000000,4.00000,10.0,0,0.870625,1.791009
owners_lower,9692.0,32220.387949,333407.472154,0.0,0.000000,0.000000,20000.00000,20000000.0,0,37.329546,1754.736105
owners_lower_calc,9692.0,32220.387949,333407.472154,0.0,0.000000,0.000000,20000.00000,20000000.0,0,37.329546,1754.736105



steam_indie_9692의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
6,ccu,9692,0,0.0,0.000000,0.000000,1.00000,83936.0,1.000000,-1.500000,2.500000,1864,19.23
3,total_reviews_calc,9692,0,10.0,18.000000,41.000000,152.00000,370046.0,134.000000,-183.000000,353.000000,1501,15.49
2,total_reviews,9692,0,10.0,18.000000,41.000000,152.00000,370046.0,134.000000,-183.000000,353.000000,1501,15.49
0,positive,9692,0,0.0,15.000000,34.000000,127.00000,327889.0,112.000000,-153.000000,295.000000,1500,15.48
1,negative,9692,0,0.0,2.000000,6.000000,22.00000,106084.0,20.000000,-28.000000,52.000000,1448,14.94
11,owners_mid_calc,9692,0,10000.0,10000.000000,10000.000000,35000.00000,35000000.0,25000.000000,-27500.000000,72500.000000,1255,12.95
10,owners_upper_calc,9692,0,20000.0,20000.000000,20000.000000,50000.00000,50000000.0,30000.000000,-25000.000000,95000.000000,1255,12.95
9,owners_lower_calc,9692,0,0.0,0.000000,0.000000,20000.00000,20000000.0,20000.000000,-30000.000000,50000.000000,689,7.11
8,owners_lower,9692,0,0.0,0.000000,0.000000,20000.00000,20000000.0,20000.000000,-30000.000000,50000.000000,689,7.11
4,positive_ratio_pct_calc,9692,0,0.0,77.065297,88.372093,95.34159,100.0,18.276293,49.650858,122.756029,359,3.70


## 메인 기술통계 3: `steam_app_details`

게임 상세 메타데이터 테이블\
장르, 카테고리, 가격, 플랫폼, 출시일 등 본 분석에 필요한 상세 정보를 제공

In [106]:
check_basic_info(app_details_df, "steam_app_details")
check_id_duplicates(app_details_df, "appid", "steam_app_details")
check_string_space_summary(app_details_df, "steam_app_details")


steam_app_details의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,161860
1,열 개수,29
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
metacritic_url,str,4093,2.53,157767,97.47,4060
metacritic_score,float64,4093,2.53,157767,97.47,72
initial_formatted,str,9895,6.11,151965,93.89,489
recommendations_total,float64,21111,13.04,140749,86.96,5316
controller_support,str,41411,25.58,120449,74.42,1
website,str,64726,39.99,97134,60.01,52441
achievements_total,float64,66565,41.13,95295,58.87,436
final_formatted,str,99878,61.71,61982,38.29,2190
final,float64,99878,61.71,61982,38.29,1985
initial,float64,99878,61.71,61982,38.29,1475


[상위 5행]


,appid,name,type,is_free,controller_support,short_description,supported_languages,developers,publishers,genres,categories,coming_soon,release_date,currency,initial,final,discount_percent,initial_formatted,final_formatted,windows,mac,linux,recommendations_total,metacritic_score,metacritic_url,achievements_total,header_image,website,collected_at
0,410110,12 is Better Than 6,game,False,full,12 is Better Than 6 is a dynamic top-down shoo...,"English, German, Russian, French, Spanish - Sp...",Ink Stains Games,HypeTrain Digital,"Action, Indie","Single-player, Steam Achievements, Full contro...",False,"20 Nov, 2015",KRW,1050000.0,210000.0,80.0,"₩ 10,500","₩ 2,100",True,True,True,4278.0,74.0,https://www.metacritic.com/game/pc/12-is-bette...,46.0,https://shared.akamai.steamstatic.com/store_it...,https://inkstainsgames.net/,2026-04-16 06:15:00.085538 +00:00
1,410120,AGON - The Mysterious Codex (Trilogy),game,False,NaN,Professor Hunt embarks on an adventure journey...,"English<strong>*</strong>, German, French, Ita...",Private Moon Studios,M.INDIE,Adventure,"Single-player, Steam Trading Cards, Family Sha...",False,"18 Nov, 2015",KRW,1050000.0,1050000.0,0.0,NaN,"₩ 10,500",True,False,False,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,2026-04-16 06:15:00.085538 +00:00
2,410130,AGON - The Lost Sword of Toledo,game,False,NaN,Search the Spanish city of Toledo with Profess...,"English<strong>*</strong>, German<strong>*</st...",Private Moon Studios,M.INDIE,Adventure,"Single-player, Steam Trading Cards, Family Sha...",False,"19 Nov, 2015",KRW,1050000.0,1050000.0,0.0,NaN,"₩ 10,500",True,False,False,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,2026-04-16 06:15:00.085538 +00:00
3,410150,Swapperoo,game,False,full,The friendly puzzle game intent on your defeat.,English<strong>*</strong><br><strong>*</strong...,Fallen Tree Games Ltd,Fallen Tree Games Ltd,"Casual, Indie","Single-player, Steam Achievements, Full contro...",False,"18 Dec, 2015",KRW,550000.0,550000.0,0.0,NaN,"₩ 5,500",True,False,False,NaN,NaN,NaN,23.0,https://shared.akamai.steamstatic.com/store_it...,http://www.fallentreegames.com/swapperoo/,2026-04-16 06:15:00.085538 +00:00
4,410210,Ampersand,game,False,NaN,Ampersand is a furiously fast paced racing gam...,English,PiGravity,Back To Basics Gaming,"Indie, Racing","Single-player, Steam Trading Cards, Family Sha...",False,"Oct 19, 2015",USD,199.0,199.0,0.0,NaN,$1.99,True,False,False,581.0,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.backtobasicsgaming.com/,2026-04-16 06:15:00.085538 +00:00



steam_app_details의 appid 값 중복 확인
전체 행 수: 161860
appid 고유 개수: 161860
중복 appid 개수: 0
중복 값이 없습니다.

steam_app_details의 문자열 공백/빈값 확인


C:\Users\joon5\AppData\Local\Temp\ipykernel_40888\2408928782.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include=["object"]).columns.tolist()


,컬럼명,문자열 행 수,빈 문자열/공백값 개수,앞뒤 공백 개수,연속 공백 포함 개수
6,publishers,161053,9,1644,119
5,developers,161616,0,606,103
3,short_description,161736,0,332,303
0,name,161839,0,185,244
15,website,64726,0,18,0
4,supported_languages,161692,0,0,4
1,type,161860,0,0,0
2,controller_support,41411,0,0,0
7,genres,161695,0,0,0
8,categories,161833,0,0,0


In [107]:
check_many_categories(
    app_details_stat_df,
    "steam_app_details",
    cols=[
        "type", "is_free", "coming_soon",
        "windows", "mac", "linux",
        "controller_support", "currency"
    ],
    top_n=20
)

check_structure_parse_summary(app_details_df, "steam_app_details", "genres", parse_type="list")
check_structure_parse_summary(app_details_df, "steam_app_details", "categories", parse_type="list")
check_date_parse_summary(app_details_df, "steam_app_details", "release_date")



steam_app_details의 type 범주 확인
전체 행 수: 161860
type 고유값 개수(결측 포함): 1



,type,개수,비율(%)
0,game,161860,100.0



steam_app_details의 is_free 범주 확인
전체 행 수: 161860
is_free 고유값 개수(결측 포함): 2



,is_free,개수,비율(%)
0,False,141166,87.21
1,True,20694,12.79



steam_app_details의 coming_soon 범주 확인
전체 행 수: 161860
coming_soon 고유값 개수(결측 포함): 2



...

0    Single-player, Steam Achievements, Full contro...
1    Single-player, Steam Trading Cards, Family Sha...
2    Single-player, Steam Trading Cards, Family Sha...
3    Single-player, Steam Achievements, Full contro...
4    Single-player, Steam Trading Cards, Family Sha...
5    Single-player, Multi-player, Co-op, Shared/Spl...
6    Multi-player, PvP, Online PvP, Co-op, Online C...
7    Single-player, Multi-player, PvP, Online PvP, ...
8    Single-player, Steam Achievements, Steam Tradi...
9    Single-player, Co-op, Tracked Controller Suppo...
Name: categories, dtype: str


steam_app_details의 release_date 날짜 파싱 가능성 확인


,항목,값
0,전체 행 수,161860
1,결측 제외 행 수,161577
2,날짜 변환 성공 수,125231
3,날짜 변환 실패 수,36346
4,날짜 변환 성공률(%),77.51
5,변환 가능 최소 날짜,1997-06-30 00:00:00
6,변환 가능 최대 날짜,9998-12-31 00:00:00


[날짜 변환 실패 예시]


32          Coming soon
306     To be announced
671             Q4 2026
3662            Q1 2027
6247      28. zář. 2017
8492            Q3 2026
8807              Maybe
8910            Q2 2026
9104    2017 年 8 月 17 日
9618            Q3 2027
Name: release_date, dtype: str

In [108]:
check_numeric_summary(
    app_details_stat_df,
    "steam_app_details",
    cols=[
        "initial", "final", "discount_percent",
        "recommendations_total", "metacritic_score",
        "achievements_total", "genre_count_calc", "category_count_calc"
    ]
)

check_iqr_outlier_summary(
    app_details_stat_df,
    "steam_app_details",
    cols=[
        "initial", "final", "discount_percent",
        "recommendations_total", "metacritic_score",
        "achievements_total", "genre_count_calc", "category_count_calc"
    ]
)


steam_app_details의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
initial,99878.0,1.006315e+06,1.858848e+06,79.0,330000.0,560000.0,1100000.0,85000000.0,61982,12.207624,266.156266
final,99878.0,9.390915e+05,1.797307e+06,49.0,230000.0,560000.0,1100000.0,85000000.0,61982,12.722569,292.539194
discount_percent,99878.0,5.553275e+00,1.824565e+01,0.0,0.0,0.0,0.0,100.0,61982,3.348476,10.170026
recommendations_total,21111.0,5.152554e+03,4.903235e+04,101.0,184.0,405.0,1384.0,5035848.0,140749,59.228943,5430.540781
metacritic_score,4093.0,7.384559e+01,1.024218e+01,6.0,68.0,76.0,81.0,97.0,157767,-1.030485,1.903046
achievements_total,66565.0,3.490333e+01,2.022529e+02,0.0,10.0,18.0,32.0,10979.0,95295,24.570976,693.684357
genre_count_calc,161860.0,2.901810e+00,1.327992e+00,0.0,2.0,3.0,4.0,19.0,0,0.808568,1.360456
category_count_calc,161860.0,4.761887e+00,3.164951e+00,0.0,2.0,4.0,6.0,32.0,0,1.581274,3.157157



steam_app_details의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
2,discount_percent,99878,61982,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,9895,9.91
0,initial,99878,61982,79.0,330000.0,560000.0,1100000.0,85000000.0,770000.0,-825000.0,2255000.0,6078,6.09
1,final,99878,61982,49.0,230000.0,560000.0,1100000.0,85000000.0,870000.0,-1075000.0,2405000.0,5197,5.20
7,category_count_calc,161860,0,0.0,2.0,4.0,6.0,32.0,4.0,-4.0,12.0,4783,2.96
5,achievements_total,66565,95295,0.0,10.0,18.0,32.0,10979.0,22.0,-23.0,65.0,4672,7.02
3,recommendations_total,21111,140749,101.0,184.0,405.0,1384.0,5035848.0,1200.0,-1616.0,3184.0,3089,14.63
6,genre_count_calc,161860,0,0.0,2.0,3.0,4.0,19.0,2.0,-1.0,7.0,698,0.43
4,metacritic_score,4093,157767,6.0,68.0,76.0,81.0,97.0,13.0,48.5,100.5,92,2.25


# 7. 메인 테이블 조인 가능성 확인

```text
steam_indie_list
    ↓ 필터링 결과
steam_indie_9692
    ↓ appid 기준 left join
steam_app_details
```

In [109]:
# steam_indie_9692가 steam_indie_list의 부분집합인지 확인
check_join_summary(
    indie_9692_df,
    indie_list_df,
    left_name="steam_indie_9692",
    right_name="steam_indie_list",
    key="appid"
)

# 본 분석 후보군에 app_details가 얼마나 붙는지 확인
check_join_summary(
    indie_9692_df,
    app_details_df,
    left_name="steam_indie_9692",
    right_name="steam_app_details",
    key="appid"
)

# 전체 모집단 기준 app_details 매칭률도 참고 확인
check_join_summary(
    indie_list_df,
    app_details_df,
    left_name="steam_indie_list",
    right_name="steam_app_details",
    key="appid"
)


steam_indie_9692 → steam_indie_list 조인 가능성 확인


,항목,값
0,steam_indie_9692 고유 appid 수,9692.0
1,steam_indie_list 고유 appid 수,61266.0
2,매칭되는 key 수,9692.0
3,steam_indie_9692 기준 미매칭 key 수,0.0
4,steam_indie_9692 기준 매칭률(%),100.0



steam_indie_9692 → steam_app_details 조인 가능성 확인


,항목,값
0,steam_indie_9692 고유 appid 수,9692.00
1,steam_app_details 고유 appid 수,161860.00
2,매칭되는 key 수,9548.00
3,steam_indie_9692 기준 미매칭 key 수,144.00
4,steam_indie_9692 기준 매칭률(%),98.51


[steam_indie_9692에는 있지만 steam_app_details에는 없는 appid 예시]


,appid
0,1463740
1,2837610
2,2633340
3,2754230
4,2304660
5,2597670
6,2398500
7,2742250
8,1006710
9,2588110



steam_indie_list → steam_app_details 조인 가능성 확인


,항목,값
0,steam_indie_list 고유 appid 수,61266.00
1,steam_app_details 고유 appid 수,161860.00
2,매칭되는 key 수,58453.00
3,steam_indie_list 기준 미매칭 key 수,2813.00
4,steam_indie_list 기준 매칭률(%),95.41


[steam_indie_list에는 있지만 steam_app_details에는 없는 appid 예시]


,appid
0,1119430
1,1393660
2,777900
3,745840
4,2304660
5,303530
6,1082240
7,1141870
8,435600
9,863420


In [110]:
# 실제 left join 시 행 수가 유지되는지 확인
main_join_check_df = indie_9692_df.merge(
    app_details_df,
    on="appid",
    how="left",
    suffixes=("_indie", "_details"),
    indicator=True
)

print("left join 결과 shape:", main_join_check_df.shape)
display(main_join_check_df["_merge"].value_counts(dropna=False).reset_index())

print("[app_details가 붙지 않은 행 예시]")
display(main_join_check_df[main_join_check_df["_merge"] == "left_only"].head(10))

left join 결과 shape: (9692, 43)


,_merge,count
0,both,9548
1,left_only,144
2,right_only,0


[app_details가 붙지 않은 행 예시]


,appid,name_indie,owners,positive,negative,price,ccu,genres_indie,release_date_indie,developers_indie,total_reviews,owners_lower,is_f2p,is_early_access,name_details,type,is_free,controller_support,short_description,supported_languages,developers_details,publishers,genres_details,categories,coming_soon,release_date_details,currency,initial,final,discount_percent,initial_formatted,final_formatted,windows,mac,linux,recommendations_total,metacritic_score,metacritic_url,achievements_total,header_image,website,collected_at,_merge
85,2381590,not available,"500,000 .. 1,000,000",11002,4035,0,22,"['Adventure', 'Casual', 'Indie', 'Simulation']",2023-05-24,Indiesolodev,15037,500000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
120,402710,Osiris: New Dawn,"500,000 .. 1,000,000",8064,6961,0,13,"['Action', 'Adventure', 'Indie', 'RPG']",2023-01-18,Fenix Fire Entertainment,15025,500000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
173,3146110,Hit of Chernobyl,"200,000 .. 500,000",22,10,0,0,"['Action', 'Adventure', 'Indie']",2024-09-05,Really Skinny Nerd,32,200000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
235,1242700,墲人之境-无人之境,"200,000 .. 500,000",1472,359,0,0,"['Action', 'Adventure', 'Casual', 'Indie', 'Ma...",2023-12-06,无人之境,1831,200000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
264,2651330,Intravenous 2: Mercenarism,"200,000 .. 500,000",2507,152,0,5,"['Action', 'Adventure', 'Indie', 'Simulation',...",2024-01-19,Explosive Squat Games,2659,200000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
317,1863750,挂机神话,"200,000 .. 500,000",1558,164,199,39,"['Casual', 'Indie', 'RPG', 'Strategy']",2023-04-06,摸鱼猫工作室,1722,200000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
453,2370570,Standable: Full Body Estimation,"100,000 .. 200,000",988,236,1999,389,"['Indie', 'Utilities']",2023-05-08,Standable,1224,100000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
469,2431660,Let's School Homeroom,"100,000 .. 200,000",955,64,0,0,"['Casual', 'Indie', 'Simulation']",2023-06-20,Pathea Games,1019,100000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
720,2073540,Night Gate,"50,000 .. 100,000",15,12,0,0,"['Action', 'Adventure', 'Indie', 'RPG']",2023-05-18,DangerousBob Studio LLC,27,50000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
772,2837610,CONFUSION,"50,000 .. 100,000",12,7,0,1,"['Indie', 'Simulation']",2024-04-30,TrueGames,19,50000,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


# 보조 확인
`steam_indie_reviews`, `steam_indie_tags`는 미니 분석용 데이터로\
상세 기술통계를 진행하지 않고, 타입, 기본 구조, `appid` 기준 조인 가능성만 확인

In [111]:
def check_aux_type_summary(df, df_name, key_cols=None):
    """보조 데이터는 상세 기술통계 대신 타입과 주요 키 구조만 간단히 확인한다."""
    print(f"\n{'='*80}")
    print(f"{df_name}의 타입/키 구조 간단 확인")
    print(f"{'='*80}")

    type_df = pd.DataFrame({
        "데이터타입": df.dtypes.astype(str),
        "결측치 개수": df.isnull().sum(),
        "고유값 개수": df.nunique(dropna=True)
    })

    display(type_df)

    if key_cols:
        rows = []
        for col in key_cols:
            if col in df.columns:
                rows.append({
                    "키 컬럼": col,
                    "데이터타입": str(df[col].dtype),
                    "전체 행 수": len(df),
                    "고유값 수": df[col].nunique(dropna=True)
                })
        display(pd.DataFrame(rows))


## 보조 확인 1: `steam_indie_reviews`

타입, 리뷰 ID 중복, appid 조인 가능성 만 확인

In [112]:
check_aux_type_summary(
    reviews_df,
    "steam_indie_reviews",
    key_cols=["appid", "recommendationid"]
)

check_join_summary(
    reviews_df,
    indie_9692_df,
    left_name="steam_indie_reviews",
    right_name="steam_indie_9692",
    key="appid"
)

check_join_summary(
    reviews_df,
    app_details_df,
    left_name="steam_indie_reviews",
    right_name="steam_app_details",
    key="appid"
)




steam_indie_reviews의 타입/키 구조 간단 확인


,데이터타입,결측치 개수,고유값 개수
recommendationid,int64,0,6553
appid,int64,0,73
language,str,0,24
review,str,2,6538
timestamp_created,int64,0,6553
timestamp_updated,int64,0,6553
voted_up,bool,0,2
votes_up,int64,0,377
votes_funny,int64,0,167
weighted_vote_score,float64,0,4023


,키 컬럼,데이터타입,전체 행 수,고유값 수
0,appid,int64,13106,73
1,recommendationid,int64,13106,6553



steam_indie_reviews → steam_indie_9692 조인 가능성 확인


,항목,값
0,steam_indie_reviews 고유 appid 수,73.00
1,steam_indie_9692 고유 appid 수,9692.00
2,매칭되는 key 수,62.00
3,steam_indie_reviews 기준 미매칭 key 수,11.00
4,steam_indie_reviews 기준 매칭률(%),84.93


[steam_indie_reviews에는 있지만 steam_indie_9692에는 없는 appid 예시]


,appid
0,1700300
1,1049590
2,1116540
3,1948800
4,2115850
5,1213300
6,2717010
7,2186320
8,2729480
9,2789810



steam_indie_reviews → steam_app_details 조인 가능성 확인


,항목,값
0,steam_indie_reviews 고유 appid 수,73.0
1,steam_app_details 고유 appid 수,161860.0
2,매칭되는 key 수,73.0
3,steam_indie_reviews 기준 미매칭 key 수,0.0
4,steam_indie_reviews 기준 매칭률(%),100.0


## 보조 확인 2: `steam_indie_tags`

In [113]:
check_aux_type_summary(
    tags_df,
    "steam_indie_tags",
    key_cols=["appid", "tags"]
)

check_join_summary(
    tags_df,
    indie_9692_df,
    left_name="steam_indie_tags",
    right_name="steam_indie_9692",
    key="appid"
)

check_join_summary(
    tags_df,
    app_details_df,
    left_name="steam_indie_tags",
    right_name="steam_app_details",
    key="appid"
)


steam_indie_tags의 타입/키 구조 간단 확인


,데이터타입,결측치 개수,고유값 개수
appid,int64,0,74
name,str,0,74
developer,str,0,74
publisher,str,0,73
owners,str,0,9
positive,int64,0,70
negative,int64,0,70
price,int64,0,27
tags,str,0,74
updated_at,str,0,74


,키 컬럼,데이터타입,전체 행 수,고유값 수
0,appid,int64,74,74
1,tags,str,74,74



steam_indie_tags → steam_indie_9692 조인 가능성 확인


,항목,값
0,steam_indie_tags 고유 appid 수,74.00
1,steam_indie_9692 고유 appid 수,9692.00
2,매칭되는 key 수,63.00
3,steam_indie_tags 기준 미매칭 key 수,11.00
4,steam_indie_tags 기준 매칭률(%),85.14


[steam_indie_tags에는 있지만 steam_indie_9692에는 없는 appid 예시]


,appid
0,1700300
1,1049590
2,1116540
3,1948800
4,2115850
5,1213300
6,2717010
7,2186320
8,2729480
9,2789810



steam_indie_tags → steam_app_details 조인 가능성 확인


,항목,값
0,steam_indie_tags 고유 appid 수,74.0
1,steam_app_details 고유 appid 수,161860.0
2,매칭되는 key 수,74.0
3,steam_indie_tags 기준 미매칭 key 수,0.0
4,steam_indie_tags 기준 매칭률(%),100.0


# 9. 기술통계 보고서

## 9.1 점검 목적

본 분석에 들어가기 전, 원천 데이터의 구조와 품질을 확인하기 위한 사전 점검\ 
따라서 실제 전처리를 완료하는 것이 아닌, **어떤 컬럼을 정리해야 하는지, 어떤 테이블을 기준으로 조인해야 하는지, 분석 전에 어떤 문제가 있는지 확인하는 것**을 목적으로 한다.

이번 점검에서는 메인 분석에 직접 사용될 가능성이 높은 `steam_indie_list`, `steam_indie_9692`, `steam_app_details`를 중심으로 진행\
`steam_indie_reviews`, `steam_indie_tags`는 미니 분석 단계에서 수집한 보조 데이터이므로, 본 분석에서는 상세 기술통계 대상에서 제외하고 타입과 조인 가능성만 간단히 확인하였다.


## 9.2 점검 대상 데이터

| 구분 | 파일명 | 역할 | 점검 수준 |
|---|---|---|---|
| 메인 | `steam_indie_list_202604211615.csv` | 전체 인디게임 모집단 | 상세 점검 |
| 메인 | `steam_indie_9692_202604281029.csv` | 본 분석 후보군 | 상세 점검 |
| 메인 | `steam_app_details_202604281555.csv` | 게임 상세 메타데이터 | 상세 점검 |
| 보조 | `steam_indie_reviews_202604230927.csv` | 미니 리뷰 분석용 데이터 | 타입/조인만 확인 |
| 보조 | `steam_indie_tags_202604281545.csv` | 미니 태그 분석용 데이터 | 타입/조인만 확인 |

## 9.3 메인 데이터 점검 결과

### `steam_indie_list`
`steam_indie_list`는 전체 인디게임 모집단 역할을 하는 데이터

| 항목 | 결과 |
|------------------|------:|
| 행 수             | 61,266 |
| 열 수             | 12 |
| 완전 중복 행      | 0 |
| `appid` 고유값 수 | 61,266 |
| `appid` 중복      | 0 |

`appid`가 전체 행 수와 동일하므로, 이 데이터는 **게임 1개당 1행 구조**로 정리되어 있다.  
따라서 전체 모집단 기준 테이블로 사용하기에 적절하다.

#### 결측치
| 컬럼 | 결측치 수 | 결측 비율 |
|--------------|---:|---:|
| `developers` | 96 | 0.16% |
| `release_date` | 33 | 0.05% |
| `spy_name`    | 8 | 0.01% |
| `name_store` | 3 | 0.00% |

결측 비율은 전체적으로 낮다.  
다만 `release_date`는 출시 시기 분석에 사용될 수 있으므로, 본 분석 전 날짜 변환 가능 여부를 확인해야 한다.

#### 공백 문제
| 컬럼 | 앞뒤 공백 | 연속 공백 | 빈 문자열/공백값 |
|---------------|---:|---:|---:|
| `spy_name`   | 37 | 99 | 0 |
| `name_store` | 29 | 97 | 1 |
| `developers` | 24 | 30 | 1 |
게임명이나 개발사명 기준 집계를 수행할 경우 같은 값이 다르게 인식될 수 있으므로, 본 분석 전 `str.strip()` 기반 정리가 필요하다.

#### 이상치 후보
| 컬럼                  | 중앙값 | 최댓값 | IQR 기준 이상치 후보 |
|------------------------|---:|---------:|------:|
| `positive`            | 17 | 1,373,979 | 9,925개 |
| `negative`            | 5 | 156,649 | 9,331개 |
| `total_reviews_calc`  | 23 | 1,409,473 | 9,808개 |
| `price_spy`           | 499 | 99,998 | 1,621개 |
| `ccu`                 | 0 | 143,870 | 11,203개 |

리뷰 수, 동시접속자 수, 가격은 강한 우측 치우침을 보인다.\
이는 Steam 인디게임 시장에서 소수의 대형 성공작과 다수의 소규모 게임이 함께 존재하기 때문으로 볼 수 있다.\
따라서 단순 삭제 대상이 아니라, 흥행 규모 차이를 보여주는 주요 특성으로 해석하는 것이 적절해 보인다.

### `steam_indie_9692`
`steam_indie_9692`는 전체 인디게임 리스트에서 본 분석 조건에 맞게 필터링된 후보군으로 볼 수 있다.

| 항목 | 결과 |
|------------------|------:|
| 행 수             | 9,692 |
| 열 수             | 14 |
| 완전 중복 행 | 0   |
| `appid` 고유값 수 | 9,692 |
| `appid` 중복 | 0  |
`appid` 중복이 없으므로, 이 데이터 역시 **게임 1개당 1행 구조**이다.\
본 분석의 중심 테이블로 사용하기에 적절하다.

#### 결측치

| 컬럼 | 결측치 수 | 결측 비율 |
|--------------|---:|---:|
| `developers` | 12 | 0.12% |
결측치는 거의 없으며, 본 분석에서 큰 문제는 없어 보인다.\
다만 개발사 기준 분석을 한다면 `developers` 결측 12개는 확인이 필요하다.

#### 공백 문제
| 컬럼 | 앞뒤 공백 | 연속 공백 | 빈 문자열/공백값 |
|---------------|---:|---:|---:|
| `developers` | 11 | 5 | 0 |
| `name`        | 7 | 17 | 0 |
공백 문제는 크지 않지만, 게임명/개발사명 기준 그룹화 전에 정리하는 것이 좋다.


#### 주요 특징
| 컬럼 | 결과 |
|-------------------|------|
| `is_f2p`          | 전부 `False` |
| `is_early_access` | 전부 `False` |
현재 후보군은 무료 게임과 얼리 액세스 게임이 제외된 상태로 보인다.\
따라서 이후 분석 결과를 해석할 때, 무료 게임과 얼리 액세스 게임은 분석 범위 밖이라는 점을 명시해야 한다.

#### 이상치 후보
| 컬럼 | 중앙값 | 최댓값 | IQR 기준 이상치 후보 |
|------------------|---:|---:|---:|
| `positive`       | 34 | 327,889 | 1,500개 |
| `negative`       | 6 | 106,084 | 1,448개 |
| `total_reviews`  | 41 | 370,046 | 1,501개 |
| `price`          | 599 | 19,999 | 229개 |
| `ccu`            | 0 | 83,936 | 1,864개 |
| `owners_lower`   | 0 | 20,000,000 | 689개 |
이 데이터에서도 리뷰 수, 소유자 수, 동시접속자 수는 우측 치우침이 강하다.\
이는 이상값이라기보다 게임별 흥행 규모 차이로 보는 것이 적절할것으로 판단된다.

### `steam_app_details`
`steam_app_details`는 게임 상세 메타데이터 테이블\
장르, 카테고리, 가격, 플랫폼, 출시일 등 본 분석에 필요한 세부 정보를 제공한다.

| 항목 | 결과 |
|---|---:|
| 행 수 | 161,860 |
| 열 수 | 29 |
| 완전 중복 행 | 0 |
| `appid` 고유값 수 | 161,860 |
| `appid` 중복 | 0 |
이 데이터는 전체 Steam 앱 상세 정보에 가까운 넓은 범위의 데이터\
따라서 본 분석에서는 `steam_indie_9692`를 기준으로 필요한 appid만 조인해서 사용하는 것이 적절

#### 결측치
| 컬럼 | 결측치 수 | 결측 비율 |
|---|---:|---:|
| `metacritic_url` | 157,767 | 97.47% |
| `metacritic_score` | 157,767 | 97.47% |
| `initial_formatted` | 151,965 | 93.89% |
| `recommendations_total` | 140,749 | 86.96% |
| `controller_support` | 120,449 | 74.42% |
| `website` | 97,134 | 60.01% |
| `achievements_total` | 95,295 | 58.87% |
| `final`, `initial`, `currency`, `discount_percent` | 61,982 | 38.29% |
`steam_app_details`에는 결측이 많은 컬럼이 존재한다.\
특히 `metacritic_score`, `controller_support`, `website`, `achievements_total` 등은 분석의 변수로 쓰기에는 결측 비율이 높다.\
가격 관련 컬럼의 결측은 무료 게임이거나 가격 정보가 없는 경우일 가능성이 있으므로, `is_free`와 함께 확인해야 한다.

#### 공백 문제
| 컬럼 | 앞뒤 공백 | 연속 공백 | 빈 문자열/공백값 |
|---|---:|---:|---:|
| `publishers` | 1,644 | 119 | 9 |
| `developers` | 606 | 103 | 0 |
| `short_description` | 332 | 303 | 0 |
| `name` | 185 | 244 | 0 |
상세 메타데이터는 문자열 컬럼이 많으므로, 개발사/배급사/게임명 기준 집계 전 공백 정리가 필요하다.


#### 날짜 컬럼
| 항목 | 결과 |
|---|---:|
| `steam_indie_list` 날짜 변환 실패 | 67개 |
| `steam_indie_9692` 날짜 변환 실패 | 0개 |
| `steam_app_details` 날짜 변환 실패 | 36,349개 |
`release_date`에는 `Coming soon`, `To be announced`, `2026`, `Q2 2026` 같은 불규칙 값이 포함되어 있다.\
따라서 본 분석에서는 출시 완료 게임만 사용하거나, 날짜 변환 가능한 값만 별도로 정리하는 과정이 필요하다.


#### 구조형 문자열
`steam_app_details`의 `genres`, `categories`는 쉼표로 구분된 문자열 형태로 저장되어 있다.

예시:
```text
Action, Indie
Single-player, Steam Achievements, Full controller support
```
따라서 본 분석에서 장르나 카테고리를 사용하려면 쉼표 기준 분리 작업이 필요하다.

## 9.5 조인 가능성 확인

본 분석의 핵심 조인은 다음 구조로 보는 것이 적절하다.

```text
steam_indie_list
    ↓ 필터링 결과
steam_indie_9692
    ↓ appid 기준 조인
steam_app_details
```

| 기준 테이블 | 대상 테이블 | 기준 appid 수 | 매칭 수 | 미매칭 수 | 매칭률 |
|---|---|---:|---:|---:|---:|
| `steam_indie_9692` | `steam_indie_list` | 9,692 | 9,692 | 0 | 100.00% |
| `steam_indie_9692` | `steam_app_details` | 9,692 | 9,548 | 144 | 98.51% |
| `steam_indie_list` | `steam_app_details` | 61,266 | 58,453 | 2,813 | 95.41% |

`steam_indie_9692`는 `steam_indie_list`에 모두 포함되어 있으므로, 원본 모집단에서 필터링된 데이터로 보는 것이 타당하다.

또한 `steam_indie_9692`와 `steam_app_details`는 98.51% 매칭되므로, 본 분석에 필요한 메타데이터는 대부분 조인 가능하다.\
다만 144개 게임은 상세 메타데이터가 붙지 않으므로, 본 분석 전 미매칭 원인을 확인해야 한다.

## 9.6 보조 데이터 확인

### `steam_indie_reviews`
| 항목 | 결과 |
|---|---:|
| 행 수 | 13,106 |
| 열 수 | 21 |
| 고유 appid 수 | 73 |
| `steam_indie_9692`와 매칭되는 appid 수 | 62 |
| `steam_indie_9692` 기준 매칭률 | 84.93% |
| `steam_app_details` 기준 매칭률 | 100.00% |

### `steam_indie_tags`
| 항목 | 결과 |
|---|---:|
| 행 수 | 74 |
| 열 수 | 10 |
| 고유 appid 수 | 74 |
| `steam_indie_9692`와 매칭되는 appid 수 | 63 |
| `steam_indie_9692` 기준 매칭률 | 85.14% |
| `steam_app_details` 기준 매칭률 | 100.00% |

## 9.7 본 분석 전 필요한 전처리 항목

### 필수 전처리
| 대상 | 전처리 항목 | 이유 |
|---|---|---|
| `steam_indie_list` | `release_date` 날짜 변환 가능 여부 확인 | 출시일 기준 필터링/분석 필요 |
| `steam_indie_list` | `genres` 리스트 파싱 | 장르별 분석 필요 |
| `steam_indie_list` | 게임명/개발사명 앞뒤 공백 제거 | 이름 기준 집계 오류 방지 |
| `steam_indie_9692` | `developers` 결측 확인 | 개발사 기준 분석 시 필요 |
| `steam_indie_9692` | `genres` 리스트 파싱 | 장르별 분석 필요 |
| `steam_app_details` | `release_date` 정리 | `Coming soon`, `To be announced`, 분기 표기 등 처리 필요 |
| `steam_app_details` | `genres`, `categories` 쉼표 기준 분리 | 장르/플레이 방식 분석 필요 |
| `steam_app_details` | `currency`, `initial`, `final` 가격 단위 확인 | 가격 분석 시 단위 혼동 방지 |
| 조인 결과 | appdetails 미매칭 144개 확인 | 본 분석 누락 여부 판단 |

### 선택 전처리
| 대상 | 전처리 항목 | 판단 |
|---|---|---|
| `steam_indie_reviews` | 리뷰 ID 기준 중복 제거 | 리뷰 본문 분석을 본 분석에 포함할 경우만 필요 |
| `steam_indie_reviews` | timestamp 변환 | 리뷰 시계열 분석을 할 경우만 필요 |
| `steam_indie_tags` | `tags` JSON 파싱 | 태그 분석을 본 분석에 포함할 경우만 필요 |